In [ ]:
import torch
# 核心：GPU（CUDA）优先检测，无则使用CPU
if torch.cuda.is_available():
    device = "cuda"
    # 打印GPU信息（验证+直观确认）
    gpu_count = torch.cuda.device_count()
    gpu_name = torch.cuda.get_device_name(0)
    print(f"✅ 检测到CUDA GPU，共{gpu_count}块，使用第1块：{gpu_name}")
else:
    device = "cpu"
    print("⚠️  未检测到CUDA GPU，将使用CPU运行（速度较慢）")

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from anndata import AnnData
import scanpy as sc
import squidpy as sq
import pandas as pd
import numpy as np
import copy
import os
import matplotlib.pyplot as plt
import seaborn as sns
color_list=['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd','#8c564b','#e377c2','#7f7f7f','#bcbd22','#17becf',
  '#a6cee3','#fdbf6f','#b2df8a','#fb9a99','#cab2d6','#f3cb29','#a30059','#22a884','#ff34ff','#809693','#ffdbe5','#023fa5','#9cded6']
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['font.sans-serif'] = 'Arial'

# 初步检查

In [ ]:
import scanpy as sc

In [ ]:
adata_st=sc.read("/path/to/project/20260427_AS_analysis/adatas/G21-7_repaired_AS.h5ad")
adata_st

In [ ]:
sc.pl.spatial(adata_st, img_key=None,spot_size=25, 
              color=['total_counts','n_genes_by_counts',"brain_cluster"], cmap='Spectral_r',  vmin='p1',vmax='p99',ncols=3,
              basis='spatial' ,wspace=0.2,
              show=True)

In [ ]:
adata_sc=sc.read("/path/to/project/Public_data/cell2location/adata_raw_sc.h5ad")
adata_sc

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

# ─────────────────────────────────────────
# 1. 加载数据
# ─────────────────────────────────────────
adata_st = sc.read("/path/to/project/20260427_AS_analysis/adatas/G21-7_repaired_AS.h5ad")
adata_sc = sc.read("/path/to/project/Public_data/cell2location/adata_raw_sc.h5ad")

print("=" * 60)
print("ST shape:", adata_st.shape)
print("SC shape:", adata_sc.shape)

# ─────────────────────────────────────────
# 2. ST：检查 counts layer 是否是原始整数计数
# ─────────────────────────────────────────
print("\n" + "=" * 60)
print("【ST counts layer 检查】")
import scipy.sparse as sp

X_counts = adata_st.layers['counts']
if sp.issparse(X_counts):
    sample_vals = X_counts[:10, :10].toarray()
else:
    sample_vals = X_counts[:10, :10]

print("前10×10值（应为非负整数）:")
print(sample_vals)
print("是否全为整数:", np.all(sample_vals == np.floor(sample_vals)))
print("最小值:", X_counts.min(), "  最大值:", X_counts.max())
print("是否有负值:", (X_counts.min() < 0))

# ─────────────────────────────────────────
# 3. SC：检查 X 是否是原始整数计数
# ─────────────────────────────────────────
print("\n" + "=" * 60)
print("【SC X 检查】")
X_sc = adata_sc.X
if sp.issparse(X_sc):
    sample_sc = X_sc[:10, :10].toarray()
else:
    sample_sc = X_sc[:10, :10]

print("前10×10值（应为非负整数）:")
print(sample_sc)
print("是否全为整数:", np.all(sample_sc == np.floor(sample_sc)))
print("最小值:", X_sc.min(), "  最大值:", X_sc.max())

# ─────────────────────────────────────────
# 4. 基因名对齐检查：ST 与 SC 用哪个 index
# ─────────────────────────────────────────
print("\n" + "=" * 60)
print("【基因名检查】")
print("ST var_names 前5:", list(adata_st.var_names[:5]))
print("ST var['Symbol'] 前5:", list(adata_st.var['Symbol'][:5]))
print("SC var_names 前5:", list(adata_sc.var_names[:5]))
print("SC var['Symbol'] 前5:", list(adata_sc.var['Symbol'][:5]))

# 检查 ST var_names 与 SC var_names 直接交集
overlap_direct = len(np.intersect1d(adata_st.var_names, adata_sc.var_names))
# 检查 ST Symbol 与 SC var_names 的交集
overlap_st_symbol = len(np.intersect1d(adata_st.var['Symbol'], adata_sc.var_names))
# 检查 ST var_names 与 SC Symbol 的交集
overlap_sc_symbol = len(np.intersect1d(adata_st.var_names, adata_sc.var['Symbol']))
# 检查 Symbol 与 Symbol
overlap_both_symbol = len(np.intersect1d(adata_st.var['Symbol'], adata_sc.var['Symbol']))

print(f"\nST var_names  × SC var_names  交集: {overlap_direct}")
print(f"ST Symbol     × SC var_names  交集: {overlap_st_symbol}")
print(f"ST var_names  × SC Symbol     交集: {overlap_sc_symbol}")
print(f"ST Symbol     × SC Symbol     交集: {overlap_both_symbol}  ← 应选最大的那个")

# ─────────────────────────────────────────
# 5. SC：检查 annotation_1 细胞类型分布
# ─────────────────────────────────────────
print("\n" + "=" * 60)
print("【SC 细胞类型分布（annotation_1）】")
ct_counts = adata_sc.obs['annotation_1'].value_counts()
print(ct_counts)
print(f"\n细胞类型总数: {len(ct_counts)}")
print(f"是否含 Unk/LowQ（应已过滤）: { any(ct_counts.index.str.contains('Unk|LowQ')) }")

# ─────────────────────────────────────────
# 6. SC：检查 batch/sample 信息
# ─────────────────────────────────────────
print("\n" + "=" * 60)
print("【SC sample 分布】")
print(adata_sc.obs['sample'].value_counts())

# ─────────────────────────────────────────
# 7. ST：检查 batch_key 候选列
# ─────────────────────────────────────────
print("\n" + "=" * 60)
print("【ST sample_id 分布（用于 batch_key）】")
print(adata_st.obs['sample_id'].value_counts())

# ─────────────────────────────────────────
# 8. 稀疏格式检查（cell2location 要求 CSR）
# ─────────────────────────────────────────
print("\n" + "=" * 60)
print("【稀疏矩阵格式】")
print("ST counts 格式:", type(adata_st.layers['counts']))
print("SC X 格式:      ", type(adata_sc.X))

# ─────────────────────────────────────────
# 9. 汇总：是否需要预处理操作
# ─────────────────────────────────────────
print("\n" + "=" * 60)
print("【待确认操作清单】")
print("□ ST 是否需要将 counts layer 复制到 .X")
print("□ ST/SC 基因名是否需要统一（用 Symbol 还是 Ensembl ID）")
print("□ SC 是否已过滤 Unk/LowQ 细胞")
print("□ SC X 是否为原始 counts（非 normalized）")
print("□ SC 是否需要再做 filter_genes（min_cells=10）")


In [ ]:
import scvi, cell2location, lightning
print("scvi:", scvi.__version__)
print("cell2location:", cell2location.__version__)
print("lightning:", lightning.__version__)

# cell2location

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import scanpy as sc
import pandas as pd
import numpy as np
import scipy.sparse as sp
import cell2location
from cell2location.utils.filtering import filter_genes
from cell2location.models import RegressionModel
from os import makedirs
from os.path import exists

# ─────────────────────────────────────────────────────────
# 0. 路径设置
# ─────────────────────────────────────────────────────────
result_folder = '/path/to/project/20260427_AS_analysis/cell2location_result'
ref_run_name  = f'{result_folder}/reference_signatures'
run_name_st   = f'{result_folder}/cell2location_map_G21-7'

for p in [ref_run_name, run_name_st]:
    if not exists(p):
        makedirs(p)

# ─────────────────────────────────────────────────────────
# 1. 加载数据
# ─────────────────────────────────────────────────────────
adata_st = sc.read("/path/to/project/20260427_AS_analysis/adatas/G21-7_repaired_AS.h5ad")
adata_sc = sc.read("/path/to/project/Public_data/cell2location/adata_raw_sc.h5ad")

# ─────────────────────────────────────────────────────────
# 2. ST：将 counts layer 设为 .X（cell2location 要求原始整数）
# ─────────────────────────────────────────────────────────
adata_st.X = adata_st.layers['counts'].copy()
print("ST .X 替换完成，max:", adata_st.X.max(), "  is_integer:", np.all(adata_st.X.data == np.floor(adata_st.X.data)))

# ─────────────────────────────────────────────────────────
# 3. SC：过滤 Unk 细胞（Ext_Unk_1/2/3 保留与否取决于你的研究问题）
#    这里选择过滤，与 notebook 原始做法一致
# ─────────────────────────────────────────────────────────
unk_mask = adata_sc.obs['annotation_1'].str.contains('Unk|LowQ')
print(f"过滤前 SC: {adata_sc.shape}，含 Unk/LowQ: {unk_mask.sum()} 个细胞")
adata_sc = adata_sc[~unk_mask].copy()
print(f"过滤后 SC: {adata_sc.shape}")

# ─────────────────────────────────────────────────────────
# 4. 基因对齐：两者 var_names 均为 Ensembl ID，直接用 var_names 求交集
#    同时去除线粒体基因（以 mt- 开头的 Symbol）
# ─────────────────────────────────────────────────────────
# ST 去线粒体基因（利用已有 mt 列）
if 'mt' in adata_st.var.columns:
    adata_st = adata_st[:, ~adata_st.var['mt']].copy()
    print(f"ST 去线粒体后: {adata_st.shape}")

# SC 去线粒体基因（用 Symbol 判断）
adata_sc.var['mt'] = adata_sc.var['Symbol'].str.lower().str.startswith('mt-')
adata_sc = adata_sc[:, ~adata_sc.var['mt']].copy()
print(f"SC 去线粒体后: {adata_sc.shape}")

# ─────────────────────────────────────────────────────────
# 5. SC：基因过滤（cell2location 推荐过滤器）
# ─────────────────────────────────────────────────────────
selected = filter_genes(
    adata_sc,
    cell_count_cutoff=5,
    cell_percentage_cutoff2=0.03,
    nonz_mean_cutoff=1.12
)
adata_sc_filtered = adata_sc[:, selected].copy()
print(f"SC 基因过滤后: {adata_sc_filtered.shape}")

# ─────────────────────────────────────────────────────────
# 6. 确认基因交集
# ─────────────────────────────────────────────────────────
intersect = np.intersect1d(adata_st.var_names, adata_sc_filtered.var_names)
print(f"\n最终基因交集数: {len(intersect)}")
# 检查一下交集基因的 Symbol（可选）
print("交集基因前5（Ensembl）:", intersect[:5])
print("对应 ST Symbol:", list(adata_st.var.loc[intersect[:5], 'Symbol']))

In [ ]:
# ─────────────────────────────────────────────────────────
# 第一步：训练参考模型 RegressionModel（从 scRNA 学细胞类型特征）
# ─────────────────────────────────────────────────────────

# setup：batch_key='sample'（6个供体），labels_key='annotation_1'
RegressionModel.setup_anndata(
    adata=adata_sc_filtered,
    batch_key='sample',
    labels_key='annotation_1',
)

mod_ref = RegressionModel(adata_sc_filtered)
mod_ref.view_anndata_setup()

In [ ]:
import torch
accelerator = "gpu" if torch.cuda.is_available() else "cpu"
print("使用:", accelerator)

In [ ]:
# ── RegressionModel 训练 ──
mod_ref.train(max_epochs=250, accelerator=accelerator)
mod_ref.plot_history(0)

In [ ]:
# ── RegressionModel export_posterior ──
adata_sc_filtered = mod_ref.export_posterior(
    adata_sc_filtered,
    sample_kwargs={'num_samples': 1000, 'batch_size': 2500},
    export_slot='mod',
    add_to_varm=["means", "stds", "q05", "q95"],
    # scale_average_detection 在 0.1.4 已移除，不要传
)

mod_ref.save(ref_run_name, overwrite=True)
adata_sc_filtered.write(f"{ref_run_name}/sc.h5ad")
mod_ref.plot_QC()

In [ ]:
# ── 提取 inf_aver ──
if 'means_per_cluster_mu_fg' in adata_sc_filtered.varm.keys():
    inf_aver = adata_sc_filtered.varm['means_per_cluster_mu_fg'][
        [f'means_per_cluster_mu_fg_{i}' for i in adata_sc_filtered.uns['mod']['factor_names']]
    ].copy()
else:
    inf_aver = adata_sc_filtered.var[
        [f'means_per_cluster_mu_fg_{i}' for i in adata_sc_filtered.uns['mod']['factor_names']]
    ].copy()

inf_aver.columns = adata_sc_filtered.uns['mod']['factor_names']
print("inf_aver shape:", inf_aver.shape)
inf_aver.to_csv(f'{ref_run_name}/inf_aver.csv')

In [ ]:
# ── Cell2location setup & 训练 ──
intersect = np.intersect1d(adata_st.var_names, inf_aver.index)
print(f"用于去卷积的基因数: {len(intersect)}")

adata_st_c2l = adata_st[:, intersect].copy()
inf_aver_sub  = inf_aver.loc[intersect, :].copy()

cell2location.models.Cell2location.setup_anndata(
    adata=adata_st_c2l,
    batch_key=None
)

mod_st = cell2location.models.Cell2location(
    adata_st_c2l,
    cell_state_df=inf_aver_sub,
    N_cells_per_location=1,
    detection_alpha=20,
    detection_mean_per_sample=False
)
mod_st.view_anndata_setup()

In [ ]:
mod_st.train(
    max_epochs=30000,
    batch_size=None,
    accelerator=accelerator,
)
mod_st.plot_history(1000)

In [ ]:
# ── Cell2location export_posterior ──
adata_st_c2l = mod_st.export_posterior(
    adata_st_c2l,
    sample_kwargs={'num_samples': 1000, 'batch_size': adata_st_c2l.n_obs}
)

mod_st.save(run_name_st, overwrite=True)
adata_st_c2l.write(f"{run_name_st}/sp.h5ad")
mod_st.plot_QC()

# 查看细胞比例

In [ ]:
# ─────────────────────────────────────────────────────────
# 1. 把细胞丰度写回 obs
# ─────────────────────────────────────────────────────────
adata_st_c2l.obs[adata_st_c2l.uns['mod']['factor_names']] = \
    adata_st_c2l.obsm['q05_cell_abundance_w_sf']

factor_names = adata_st_c2l.uns['mod']['factor_names']
print("细胞类型数:", len(factor_names))
print(factor_names)

In [ ]:
# ─────────────────────────────────────────────────────────
# 2. 统计每个 spot 主导细胞类型 + 各细胞类型总丰度
# ─────────────────────────────────────────────────────────
# 每个 spot 丰度最高的细胞类型
adata_st_c2l.obs['Cell_type_top'] = \
    adata_st_c2l.obs[factor_names].idxmax(axis=1)

# 各细胞类型的总丰度（用于饼图排序）
cell_abundance_sum = adata_st_c2l.obs[factor_names].sum(0).sort_values(ascending=False)
df_cell_count = pd.DataFrame({'count': cell_abundance_sum})
print(df_cell_count)

In [ ]:
# ─────────────────────────────────────────────────────────
# 3. 配色方案
# ─────────────────────────────────────────────────────────
def rgb2hex(r, g, b):
    return f'#{r:02x}{g:02x}{b:02x}'

color_pool = [
    rgb2hex(0,158,115),   rgb2hex(213,94,0),    rgb2hex(240,228,66),
    rgb2hex(86,180,233),  rgb2hex(90,20,165),   rgb2hex(140,81,10),
    rgb2hex(197,27,125),  rgb2hex(67,147,195),  rgb2hex(254,196,7),
    rgb2hex(188,189,220), rgb2hex(127,201,127), rgb2hex(178,24,43),
    rgb2hex(53,151,143),  rgb2hex(247,252,185), rgb2hex(222,203,228),
    rgb2hex(253,218,236), rgb2hex(44,160,44),   rgb2hex(214,39,40),
    rgb2hex(148,103,189), rgb2hex(140,86,75),   rgb2hex(227,119,194),
    rgb2hex(127,127,127), rgb2hex(188,189,34),  rgb2hex(23,190,207),
    rgb2hex(31,119,180),  rgb2hex(255,127,14),  rgb2hex(174,199,232),
    rgb2hex(255,187,120), rgb2hex(152,223,138), rgb2hex(255,152,150),
    rgb2hex(197,176,213), rgb2hex(196,156,148), rgb2hex(247,182,210),
    rgb2hex(199,199,199), rgb2hex(219,219,141), rgb2hex(158,218,229),
    rgb2hex(57,59,121),   rgb2hex(82,84,163),   rgb2hex(107,110,207),
    rgb2hex(156,158,222), rgb2hex(99,121,57),   rgb2hex(140,162,82),
    rgb2hex(181,207,107), rgb2hex(140,109,49),  rgb2hex(189,158,57),
    rgb2hex(231,186,82),  rgb2hex(132,60,57),   rgb2hex(173,73,74),
    rgb2hex(214,97,107),  rgb2hex(231,150,156), rgb2hex(123,65,115),
    rgb2hex(165,81,148),  rgb2hex(206,109,189), rgb2hex(222,158,214),
]

# 确保颜色数量足够
n_ct = len(df_cell_count)
colors = (color_pool * ((n_ct // len(color_pool)) + 1))[:n_ct]
df_cell_count['color'] = colors
print(df_cell_count[['count','color']].head(10))

In [ ]:
# 打印完整列表，方便你决定保留哪些
print("完整细胞类型丰度排序：")
print(df_cell_count[['count', 'color']].to_string())
print(f"\n总细胞类型数: {len(df_cell_count)}")
print(f"总丰度: {df_cell_count['count'].sum():.1f}")
print(f"\n各类型占比:")
for ct in df_cell_count.index:
    pct = df_cell_count.loc[ct, 'count'] / df_cell_count['count'].sum() * 100
    print(f"  {ct:30s} {pct:.2f}%")

In [ ]:
# ─────────────────────────────────────────────────────────
# 1. 定义保留的细胞类型（≥1%）
# ─────────────────────────────────────────────────────────
keep_types = [ct for ct in df_cell_count.index 
              if df_cell_count.loc[ct, 'count'] / df_cell_count['count'].sum() >= 0.01]
print(f"保留 {len(keep_types)} 个细胞类型:")
for ct in keep_types:
    pct = df_cell_count.loc[ct, 'count'] / df_cell_count['count'].sum() * 100
    print(f"  {ct:25s} {pct:.2f}%")

other_types = [ct for ct in df_cell_count.index if ct not in keep_types]
print(f"\n归入 other 的 ({len(other_types)} 个): {other_types}")

In [ ]:
# ─────────────────────────────────────────────────────────
# 2. 配色（可以修改任意色值）
# ─────────────────────────────────────────────────────────
color_assign = {
    'Ext_Thal_1'    : '#cac43d',
    'Oligo_1'       : '#10976f',
    'Ext_L23'       : '#1681ec',
    'Ext_L25'       : '#a81e2b',
    'Ext_L56'       : '#5a14a5',
    'Ext_Hpc_CA1'   : '#f0e442',
    'Ext_Thal_2'    : '#b71e75',
    'Ext_Hpc_DG1'   : '#009e73',
    'Oligo_2'       : '#67c1dd',
    'Ext_L5_1'      : '#fd8d3c',
    'Ext_Pir'       : '#74c476',
    'Inh_4'         : '#ffbe2c',
    'Micro'         : '#9e9ac8',
    'Endo'          : '#ec7c7f',
    'Ext_Amy_2'     : '#8c510a',
    'Inh_1'         : '#865123',
    'Nb_2'          : '#2ca02c',
    'Ext_Hpc_CA3'   : '#56b4e9',
    'Astro_WM'      : '#e7969c',
    'Inh_5'         : '#7b4173',
    'Inh_Pvalb'     : '#ce6dbd',
    'Ext_L6B'       : '#3182bd',
    'Inh_Meis2_3'   : '#6baed6',
    'Ext_L5_2'      : '#e6550d',
    'Ext_L6'        : '#31a354',
    'Ext_ClauPyr'   : '#756bb1',
    'Inh_6'         : '#636363',
    'Astro_HPC'     : '#d6616b',
    'Astro_HYPO'    : '#8ca252',
    'Astro_THAL_lat': '#bd9e39',
    'Astro_AMY_CTX' : '#ad494a',
    'Ext_Med'       : '#a55194',
    'Astro_AMY'     : '#6b6ecf',
    'Astro_THAL_hab': '#b5cf6b',
    'Inh_Meis2_4'   : '#9c9ede',
    'other'         : '#e0e0e0',
}

# 只保留实际存在的 keep_types 对应颜色
color_assign = {k: v for k, v in color_assign.items() if k in keep_types + ['other']}

# 更新 df_cell_count 配色
for ct in keep_types:
    df_cell_count.loc[ct, 'color'] = color_assign[ct]

In [ ]:
# ─────────────────────────────────────────────────────────
# 3. 合并 Cell_type_merged 列
# ─────────────────────────────────────────────────────────
adata_st_c2l.obs['Cell_type_merged'] = adata_st_c2l.obs['Cell_type_top'].apply(
    lambda x: x if x in keep_types else 'other'
)
cat_order = keep_types + ['other']
adata_st_c2l.obs['Cell_type_merged'] = pd.Categorical(
    adata_st_c2l.obs['Cell_type_merged'], categories=cat_order
)
print(adata_st_c2l.obs['Cell_type_merged'].value_counts())

In [ ]:
# ─────────────────────────────────────────────────────────
# 全局绘图设置（所有图统一）
# ─────────────────────────────────────────────────────────
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['pdf.fonttype'] = 42        # PDF 中字体可编辑
plt.rcParams['ps.fonttype']  = 42
plt.rcParams['font.family']  = 'Arial'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['figure.dpi']   = 150
sns.set(style="ticks", font='Arial', font_scale=1.2)

In [ ]:
# ─────────────────────────────────────────────────────────
# 饼图（PDF + Arial）
# ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 10))
wedges, texts, autotexts = ax.pie(
    pie_values,
    labels=pie_labels,
    colors=pie_colors,
    autopct=lambda p: f'{p:.1f}%' if p >= 1.0 else '',
    startangle=90,
    textprops={'fontsize': 9, 'fontfamily': 'Arial'},
    pctdistance=0.82,
    wedgeprops={'linewidth': 0.5, 'edgecolor': 'white'}
)
for at in autotexts:
    at.set_fontsize(7)
    at.set_fontfamily('Arial')
ax.set_title('Cell type abundance', fontsize=13, fontfamily='Arial')
plt.tight_layout()
plt.savefig(f'{result_folder}/celltype_pie_merged.pdf',
            dpi=300, transparent=True, format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────
# 主导细胞类型空间图（PDF + Arial）
# ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 8))
sc.pl.spatial(
    adata_st_c2l,
    img_key=None,
    spot_size=26,               # 缩小 spot
    color=['Cell_type_merged'],
    palette=palette_merged,
    ax=ax,
    show=False,
    title='Dominant cell type per spot'
)
ax.set_title('Dominant cell type per spot', fontfamily='Arial', fontsize=12)
plt.tight_layout()
plt.savefig(f'{result_folder}/celltype_spatial_merged.pdf',
            dpi=300, transparent=True, format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────
# 各细胞类型热图（PDF + Arial，每批8个）
# ─────────────────────────────────────────────────────────
for batch_i, start in enumerate(range(0, len(keep_types), 8)):
    batch = keep_types[start:start+8]
    fig, axes = plt.subplots(2, 4, figsize=(20, 9))
    axes = axes.flatten()
    for j, ct in enumerate(batch):
        sc.pl.spatial(
            adata_st_c2l,
            img_key=None,
            spot_size=15,
            color=[ct],
            cmap='Reds',
            vmin='p5', vmax='p99',
            ax=axes[j],
            show=False,
            title=ct,
            use_raw=False
        )
        axes[j].set_title(ct, fontfamily='Arial', fontsize=10)
    for j in range(len(batch), len(axes)):
        axes[j].set_visible(False)
    plt.suptitle(f'Cell type abundance (batch {batch_i+1})',
                 fontsize=12, fontfamily='Arial', y=1.01)
    plt.tight_layout()
    plt.savefig(f'{result_folder}/celltype_heatmap_batch{batch_i+1}.pdf',
                dpi=300, transparent=True, format='pdf', bbox_inches='tight')
    plt.show()

# 黑底空间图

In [ ]:
from plot_spatial_all import get_rgb_function,plot_spatial_general,plot_spatial_all,rgb_to_ryb,ryb_to_rgb,hex2rgb_3
from plot_spatial import get_rgb_function,plot_spatial_general,plot_spatial,rgb_to_ryb,ryb_to_rgb

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

# 全局字体设置
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype']  = 42
plt.rcParams['font.family']  = 'Arial'
plt.rcParams['font.sans-serif'] = ['Arial']

In [ ]:
from plot_spatial_all import plot_spatial_all, plot_Multiple, hex2rgb_3

# ─────────────────────────────────────────────────────────
# 图1：所有 keep_types 混色黑底图（用 plot_Multiple）
# 颜色顺序对应 keep_types 顺序
# ─────────────────────────────────────────────────────────
clust_labels = keep_types   # ≥1% 的细胞类型列表
clust_col    = [str(i) for i in clust_labels]
cmaps_list   = [color_assign[ct] for ct in clust_labels]  # 对应颜色

with mpl.rc_context({'figure.figsize': (12, 10)}):
    fig = plot_Multiple(
        labels=clust_labels,
        value_df=adata_st_c2l.obs[clust_col],
        coords=adata_st_c2l.obsm['spatial'],
        cmaps=cmaps_list,
        circle_diameter=2,           # 20μm 分辨率，点不要太大
        min_color_quantile=0.10,
        max_color_quantile=0.998,
        style='dark_background',
        colorbar_position='right',
        axis_y_flipped=True,
    )

plt.savefig(f'{result_folder}/cell2location_all_dark.pdf',
            dpi=300, transparent=False, format='pdf',
            facecolor='black', bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────
# 单独图例（黑底，圆点 + 细胞类型名）
# ─────────────────────────────────────────────────────────
from matplotlib.lines import Line2D

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype']  = 42
plt.rcParams['font.family']  = 'Arial'

n_ct = len(keep_types)
n_cols = 2  # 两列排列，可改成1或3
n_rows = (n_ct + n_cols - 1) // n_cols

fig, ax = plt.subplots(figsize=(n_cols * 3, n_rows * 0.35 + 0.3))
fig.patch.set_facecolor('black')
ax.set_facecolor('black')
ax.axis('off')

legend_elements = [
    Line2D([0], [0],
           marker='o',
           color='none',
           markerfacecolor=color_assign[ct],
           markeredgecolor='none',
           markersize=8,
           label=ct)
    for ct in keep_types
]

legend = ax.legend(
    handles=legend_elements,
    loc='center',
    frameon=False,
    fontsize=9,
    ncol=n_cols,
    labelcolor='white',
    handletextpad=0.5,
    columnspacing=1.0,
    borderpad=0.5,
)
for text in legend.get_texts():
    text.set_fontfamily('Arial')

plt.tight_layout(pad=0.3)
plt.savefig(f'{result_folder}/cell2location_legend.pdf',
            dpi=300, transparent=False, format='pdf',
            facecolor='black', bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
# ─────────────────────────────────────────────────────────
# 图2：plot_spatial_all 标准混色图（白底，用于对比）
# ─────────────────────────────────────────────────────────
with mpl.rc_context({'figure.figsize': (12, 10)}):
    fig = plot_spatial_all(
        adata=adata_st_c2l,
        color=clust_col,
        labels=clust_labels,
        show_img=False,
        style='fast',                # 白底
        min_color_quantile=0.10,
        max_color_quantile=0.998,
        circle_diameter=2,
        colorbar_position='right',
    )

plt.savefig(f'{result_folder}/cell2location_all_white.pdf',
            dpi=300, transparent=True, format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────
# UMAP：brain_cluster（解卷积前）+ Cell_type_merged（解卷积后）
# ─────────────────────────────────────────────────────────

# 先把 Cell_type_merged 写回原始 adata_st（它有 UMAP 坐标）
adata_st.obs['Cell_type_merged'] = adata_st_c2l.obs['Cell_type_merged']
adata_st.obs['Cell_type_top']    = adata_st_c2l.obs['Cell_type_top']

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype']  = 42
plt.rcParams['font.family']  = 'Arial'
plt.rcParams['font.sans-serif'] = ['Arial']

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# 左图：brain_cluster（解卷积前）
sc.pl.umap(
    adata_st,
    color='brain_cluster',
    ax=axes[0],
    show=False,
    title='Brain cluster (before deconvolution)',
    legend_loc='on data',
    legend_fontsize=10,
    size=20,
)
axes[0].set_title('Brain cluster (before deconvolution)', fontfamily='Arial', fontsize=11)

# 右图：Cell_type_merged（解卷积后）
sc.pl.umap(
    adata_st,
    color='Cell_type_merged',
    palette=palette_merged,
    ax=axes[1],
    show=False,
    title='Cell type (cell2location)',
    legend_loc='on data',
    legend_fontsize=10,
    size=20,
)
axes[1].set_title('Cell type (cell2location)', fontfamily='Arial', fontsize=11)

plt.tight_layout()
plt.savefig(f'{result_folder}/umap_brain_cluster_vs_celltype.pdf',
            dpi=300, transparent=True, format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
brain_color={
    'Fiber_tracts':      '#1f77b4',
    'FT':      '#1f77b4',
    'Thalamus':      '#ff7f0e',
    'TH':      '#ff7f0e',
    'Cortex_5': '#8c564b',
    'Cortex_3':    '#d62728',
    'Cortex_4':      '#22a884',
    'Cortex_2':    '#2ca02c',
    'Hypothalamus':      '#e377c2',
    'HY':      '#e377c2',
    'Cortex_1':    '#bcbd22',
    'CA1':     '#17becf',
    'DG':      '#a6cee3',
    'DG-sg':  '#6baed6', 
    'CA3':     '#b2df8a',
    'CA3_1':   '#8fcf73',
    'CA3_2':   '#c9e8b0',
    'Meninges':'#cab2d6',
    'Epithalamus':     '#f3cb29',
    'EPI':     '#f3cb29',
    'RT':      '#a30059',
    'VS':      '#9467bd',
    'HIP':     '#809693',
    'Striatum':     '#9cded6',
    'STR':     '#9cded6',
    'unknown': '#808080'
}

In [ ]:
# ── 黑底版 ──
sample="G21-7"
with mpl.style.context('dark_background'):
    fig, ax = plt.subplots(figsize=(6, 6))
    sc.pl.spatial(
        adata_st,
        img_key=None,
        spot_size=26,
        color=['brain_cluster'],
        title=f'{sample}-brain_cluster',
        basis='spatial',
        palette=brain_color,
        ax=ax,
        show=False
    )
    # 强制黑底样式
    ax.set_facecolor('black')
    fig.patch.set_facecolor('black')
    ax.set_title(f'{sample}-brain_cluster', color='white', fontfamily='Arial', fontsize=12)
    # 图例文字改白色
    legend = ax.get_legend()
    if legend:
        for text in legend.get_texts():
            text.set_color('white')
            text.set_fontfamily('Arial')
        legend.get_frame().set_facecolor('black')
        legend.get_frame().set_edgecolor('none')

    plt.tight_layout()
    plt.savefig(f'{result_folder}/G21-7_brain_cluster_dark.pdf',
                dpi=300, transparent=False, format='pdf',
                facecolor='black', bbox_inches='tight')
    plt.show()
    plt.close()

# G21-4运行

In [ ]:
("/path/to/project/20260427_AS_analysis/adatas/G21-4_anno0530.h5ad")

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

# ─────────────────────────────────────────
# 1. 加载数据
# ─────────────────────────────────────────
adata_st = sc.read("/path/to/project/20260427_AS_analysis/adatas/G21-4_anno0530.h5ad")
adata_sc = sc.read("/path/to/project/Public_data/cell2location/adata_raw_sc.h5ad")

print("=" * 60)
print("ST shape:", adata_st.shape)
print("SC shape:", adata_sc.shape)

# ─────────────────────────────────────────
# 2. ST：检查 counts layer 是否是原始整数计数
# ─────────────────────────────────────────
print("\n" + "=" * 60)
print("【ST counts layer 检查】")
import scipy.sparse as sp

X_counts = adata_st.layers['counts']
if sp.issparse(X_counts):
    sample_vals = X_counts[:10, :10].toarray()
else:
    sample_vals = X_counts[:10, :10]

print("前10×10值（应为非负整数）:")
print(sample_vals)
print("是否全为整数:", np.all(sample_vals == np.floor(sample_vals)))
print("最小值:", X_counts.min(), "  最大值:", X_counts.max())
print("是否有负值:", (X_counts.min() < 0))

# ─────────────────────────────────────────
# 3. SC：检查 X 是否是原始整数计数
# ─────────────────────────────────────────
print("\n" + "=" * 60)
print("【SC X 检查】")
X_sc = adata_sc.X
if sp.issparse(X_sc):
    sample_sc = X_sc[:10, :10].toarray()
else:
    sample_sc = X_sc[:10, :10]

print("前10×10值（应为非负整数）:")
print(sample_sc)
print("是否全为整数:", np.all(sample_sc == np.floor(sample_sc)))
print("最小值:", X_sc.min(), "  最大值:", X_sc.max())

# ─────────────────────────────────────────
# 4. 基因名对齐检查：ST 与 SC 用哪个 index
# ─────────────────────────────────────────
print("\n" + "=" * 60)
print("【基因名检查】")
print("ST var_names 前5:", list(adata_st.var_names[:5]))
print("ST var['Symbol'] 前5:", list(adata_st.var['Symbol'][:5]))
print("SC var_names 前5:", list(adata_sc.var_names[:5]))
print("SC var['Symbol'] 前5:", list(adata_sc.var['Symbol'][:5]))

# 检查 ST var_names 与 SC var_names 直接交集
overlap_direct = len(np.intersect1d(adata_st.var_names, adata_sc.var_names))
# 检查 ST Symbol 与 SC var_names 的交集
overlap_st_symbol = len(np.intersect1d(adata_st.var['Symbol'], adata_sc.var_names))
# 检查 ST var_names 与 SC Symbol 的交集
overlap_sc_symbol = len(np.intersect1d(adata_st.var_names, adata_sc.var['Symbol']))
# 检查 Symbol 与 Symbol
overlap_both_symbol = len(np.intersect1d(adata_st.var['Symbol'], adata_sc.var['Symbol']))

print(f"\nST var_names  × SC var_names  交集: {overlap_direct}")
print(f"ST Symbol     × SC var_names  交集: {overlap_st_symbol}")
print(f"ST var_names  × SC Symbol     交集: {overlap_sc_symbol}")
print(f"ST Symbol     × SC Symbol     交集: {overlap_both_symbol}  ← 应选最大的那个")

# ─────────────────────────────────────────
# 5. SC：检查 annotation_1 细胞类型分布
# ─────────────────────────────────────────
print("\n" + "=" * 60)
print("【SC 细胞类型分布（annotation_1）】")
ct_counts = adata_sc.obs['annotation_1'].value_counts()
print(ct_counts)
print(f"\n细胞类型总数: {len(ct_counts)}")
print(f"是否含 Unk/LowQ（应已过滤）: { any(ct_counts.index.str.contains('Unk|LowQ')) }")

# ─────────────────────────────────────────
# 6. SC：检查 batch/sample 信息
# ─────────────────────────────────────────
print("\n" + "=" * 60)
print("【SC sample 分布】")
print(adata_sc.obs['sample'].value_counts())

# ─────────────────────────────────────────
# 7. ST：检查 batch_key 候选列
# ─────────────────────────────────────────
print("\n" + "=" * 60)
print("【ST sample_id 分布（用于 batch_key）】")
print(adata_st.obs['sample_id'].value_counts())

# ─────────────────────────────────────────
# 8. 稀疏格式检查（cell2location 要求 CSR）
# ─────────────────────────────────────────
print("\n" + "=" * 60)
print("【稀疏矩阵格式】")
print("ST counts 格式:", type(adata_st.layers['counts']))
print("SC X 格式:      ", type(adata_sc.X))

# ─────────────────────────────────────────
# 9. 汇总：是否需要预处理操作
# ─────────────────────────────────────────
print("\n" + "=" * 60)
print("【待确认操作清单】")
print("□ ST 是否需要将 counts layer 复制到 .X")
print("□ ST/SC 基因名是否需要统一（用 Symbol 还是 Ensembl ID）")
print("□ SC 是否已过滤 Unk/LowQ 细胞")
print("□ SC X 是否为原始 counts（非 normalized）")
print("□ SC 是否需要再做 filter_genes（min_cells=10）")


In [ ]:
import warnings
warnings.filterwarnings("ignore")
import scanpy as sc
import pandas as pd
import numpy as np
import scipy.sparse as sp
import cell2location
from cell2location.utils.filtering import filter_genes
from cell2location.models import RegressionModel
from os import makedirs
from os.path import exists
import torch

# ─────────────────────────────────────────────────────────
# 路径设置
# ─────────────────────────────────────────────────────────
result_folder = '/path/to/project/20260427_AS_analysis/cell2location_result_G21-4'
ref_run_name  = f'{result_folder}/reference_signatures'
run_name_st   = f'{result_folder}/cell2location_map_G21-4'

for p in [ref_run_name, run_name_st]:
    if not exists(p): makedirs(p)

accelerator = "gpu" if torch.cuda.is_available() else "cpu"
print("使用:", accelerator)

In [ ]:
# ─────────────────────────────────────────────────────────
# 加载数据
# ─────────────────────────────────────────────────────────
adata_st = sc.read("/path/to/project/20260427_AS_analysis/adatas/G21-4_anno0530.h5ad")
adata_sc  = sc.read("/path/to/project/Public_data/cell2location/adata_raw_sc.h5ad")
print("ST:", adata_st.shape, "  SC:", adata_sc.shape)

In [ ]:
# ─────────────────────────────────────────────────────────
# 预处理
# ─────────────────────────────────────────────────────────
# ST：counts → X
adata_st.X = adata_st.layers['counts'].copy()

# SC：过滤 Unk 细胞
unk_mask = adata_sc.obs['annotation_1'].str.contains('Unk|LowQ')
adata_sc  = adata_sc[~unk_mask].copy()
print(f"SC 过滤后: {adata_sc.shape}")

# 去线粒体基因
adata_st = adata_st[:, ~adata_st.var['mt']].copy()
adata_sc.var['mt'] = adata_sc.var['Symbol'].str.lower().str.startswith('mt-')
adata_sc = adata_sc[:, ~adata_sc.var['mt']].copy()
print(f"ST 去mt后: {adata_st.shape}  SC 去mt后: {adata_sc.shape}")

In [ ]:
# ─────────────────────────────────────────────────────────
# SC 基因过滤
# ─────────────────────────────────────────────────────────

# 注意：如果 G21-7 已经训练好了 ref 模型，直接复用 inf_aver，跳过这步！
# 只有第一次才需要重新训练 RegressionModel

# 检查是否已有 inf_aver
ref_run_name_G217 = '/path/to/project/20260427_AS_analysis/cell2location_result/reference_signatures'
inf_aver_path = f'{ref_run_name_G217}/inf_aver.csv'

if exists(inf_aver_path):
    print("检测到已有 inf_aver，直接复用，跳过 RegressionModel 训练")
    inf_aver = pd.read_csv(inf_aver_path, index_col=0)
    print("inf_aver shape:", inf_aver.shape)
else:
    print("未找到 inf_aver，需要重新训练 RegressionModel")

In [ ]:
# ─────────────────────────────────────────────────────────
# 只有 inf_aver 不存在时才运行以下训练（通常可跳过）
# ─────────────────────────────────────────────────────────
if not exists(inf_aver_path):
    selected = filter_genes(adata_sc, cell_count_cutoff=5,
                            cell_percentage_cutoff2=0.03, nonz_mean_cutoff=1.12)
    adata_sc_filtered = adata_sc[:, selected].copy()

    RegressionModel.setup_anndata(adata_sc_filtered, batch_key='sample', labels_key='annotation_1')
    mod_ref = RegressionModel(adata_sc_filtered)
    mod_ref.train(max_epochs=250, accelerator=accelerator)
    mod_ref.plot_history(0)

    adata_sc_filtered = mod_ref.export_posterior(
        adata_sc_filtered,
        sample_kwargs={'num_samples': 1000, 'batch_size': 2500},
        export_slot='mod',
        add_to_varm=["means", "stds", "q05", "q95"],
    )
    mod_ref.save(ref_run_name, overwrite=True)

    if 'means_per_cluster_mu_fg' in adata_sc_filtered.varm.keys():
        inf_aver = adata_sc_filtered.varm['means_per_cluster_mu_fg'][
            [f'means_per_cluster_mu_fg_{i}' for i in adata_sc_filtered.uns['mod']['factor_names']]
        ].copy()
    else:
        inf_aver = adata_sc_filtered.var[
            [f'means_per_cluster_mu_fg_{i}' for i in adata_sc_filtered.uns['mod']['factor_names']]
        ].copy()
    inf_aver.columns = adata_sc_filtered.uns['mod']['factor_names']
    inf_aver.to_csv(f'{ref_run_name}/inf_aver.csv')
    print("inf_aver shape:", inf_aver.shape)

In [ ]:
# ─────────────────────────────────────────────────────────
# Cell2location 空间去卷积
# ─────────────────────────────────────────────────────────
intersect = np.intersect1d(adata_st.var_names, inf_aver.index)
print(f"基因交集数: {len(intersect)}")

adata_st_c2l = adata_st[:, intersect].copy()
inf_aver_sub  = inf_aver.loc[intersect, :].copy()

cell2location.models.Cell2location.setup_anndata(adata=adata_st_c2l, batch_key=None)

mod_st = cell2location.models.Cell2location(
    adata_st_c2l,
    cell_state_df=inf_aver_sub,
    N_cells_per_location=1,
    detection_alpha=20,
    detection_mean_per_sample=False
)
mod_st.view_anndata_setup()

In [ ]:
mod_st.train(max_epochs=30000, batch_size=None, accelerator=accelerator)
mod_st.plot_history(1000)

In [ ]:
adata_st_c2l = mod_st.export_posterior(
    adata_st_c2l,
    sample_kwargs={'num_samples': 1000, 'batch_size': adata_st_c2l.n_obs}
)
mod_st.save(run_name_st, overwrite=True)
adata_st_c2l.write(f"{run_name_st}/sp.h5ad")
mod_st.plot_QC()

In [ ]:
# ─────────────────────────────────────────────────────────
# 结果整理
# ─────────────────────────────────────────────────────────
factor_names = list(adata_st_c2l.uns['mod']['factor_names'])
adata_st_c2l.obs[factor_names] = adata_st_c2l.obsm['q05_cell_abundance_w_sf']
adata_st_c2l.obs['Cell_type_top'] = adata_st_c2l.obs[factor_names].idxmax(axis=1)

cell_abundance_sum = adata_st_c2l.obs[factor_names].sum(0).sort_values(ascending=False)
df_cell_count = pd.DataFrame({'count': cell_abundance_sum})

print(df_cell_count)
print(f"\n总丰度: {df_cell_count['count'].sum():.1f}")
for ct in df_cell_count.index:
    pct = df_cell_count.loc[ct, 'count'] / df_cell_count['count'].sum() * 100
    print(f"  {ct:25s} {pct:.2f}%")

## 查看结果G21-4 的细胞类型丰度

In [ ]:
# ─────────────────────────────────────────────────────────
# 结果整理 + 丰度统计
# ─────────────────────────────────────────────────────────
factor_names = list(adata_st_c2l.uns['mod']['factor_names'])
adata_st_c2l.obs[factor_names] = adata_st_c2l.obsm['q05_cell_abundance_w_sf']
adata_st_c2l.obs['Cell_type_top'] = adata_st_c2l.obs[factor_names].idxmax(axis=1)

cell_abundance_sum = adata_st_c2l.obs[factor_names].sum(0).sort_values(ascending=False)
df_cell_count = pd.DataFrame({'count': cell_abundance_sum})

print("各类型占比:")
for ct in df_cell_count.index:
    pct = df_cell_count.loc[ct, 'count'] / df_cell_count['count'].sum() * 100
    print(f"  {ct:25s} {pct:.2f}%")

## 绘图

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from plot_spatial_all import plot_spatial_all, plot_Multiple, hex2rgb_3
from matplotlib.lines import Line2D

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype']  = 42
plt.rcParams['font.family']  = 'Arial'
plt.rcParams['font.sans-serif'] = ['Arial']

sample = adata_st_c2l.obs['sample_id'][0]
print("当前样本:", sample)

# ─────────────────────────────────────────────────────────
# 1. 细胞类型合并（≥1% 保留，其余归 other）
# ─────────────────────────────────────────────────────────
keep_types_G214 = [ct for ct in df_cell_count.index
                   if df_cell_count.loc[ct, 'count'] / df_cell_count['count'].sum() >= 0.01]
print(f"保留 {len(keep_types_G214)} 个细胞类型")

# 复用 G21-7 的配色，新增类型补灰色
color_assign_G214 = {ct: color_assign.get(ct, '#aaaaaa') for ct in keep_types_G214}
color_assign_G214['other'] = '#e0e0e0'

cat_order_G214 = keep_types_G214 + ['other']
adata_st_c2l.obs['Cell_type_merged'] = adata_st_c2l.obs['Cell_type_top'].apply(
    lambda x: x if x in keep_types_G214 else 'other'
)
adata_st_c2l.obs['Cell_type_merged'] = pd.Categorical(
    adata_st_c2l.obs['Cell_type_merged'], categories=cat_order_G214
)
palette_G214 = {ct: color_assign_G214[ct] for ct in cat_order_G214}

In [ ]:
# ─────────────────────────────────────────────────────────
# 2. 饼图
# ─────────────────────────────────────────────────────────
pie_labels = keep_types_G214 + ['other']
pie_values = [df_cell_count.loc[ct, 'count'] for ct in keep_types_G214]
pie_values.append(df_cell_count.loc[~df_cell_count.index.isin(keep_types_G214), 'count'].sum())
pie_colors = [color_assign_G214[ct] for ct in pie_labels]

fig, ax = plt.subplots(figsize=(10, 10))
wedges, texts, autotexts = ax.pie(
    pie_values,
    labels=pie_labels,
    colors=pie_colors,
    autopct=lambda p: f'{p:.1f}%' if p >= 1.0 else '',
    startangle=90,
    textprops={'fontsize': 9, 'fontfamily': 'Arial'},
    pctdistance=0.82,
    wedgeprops={'linewidth': 0.5, 'edgecolor': 'white'}
)
for at in autotexts:
    at.set_fontsize(7)
    at.set_fontfamily('Arial')
ax.set_title(f'{sample} - Cell type abundance', fontsize=13, fontfamily='Arial')
plt.tight_layout()
plt.savefig(f'{result_folder}/celltype_pie_{sample}.pdf',
            dpi=300, transparent=True, format='pdf', bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
# ─────────────────────────────────────────────────────────
# 3. 主导细胞类型空间图
# ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 8))
sc.pl.spatial(
    adata_st_c2l,
    img_key=None,
    spot_size=20,
    color=['Cell_type_merged'],
    palette=palette_G214,
    ax=ax,
    show=False,
)
ax.set_title(f'{sample} - Dominant cell type', fontfamily='Arial', fontsize=12)
plt.tight_layout()
plt.savefig(f'{result_folder}/celltype_spatial_dominant_{sample}.pdf',
            dpi=300, transparent=True, format='pdf', bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
# ─────────────────────────────────────────────────────────
# 4. 黑底混色图（plot_Multiple）
# ─────────────────────────────────────────────────────────
clust_labels = keep_types_G214
clust_col    = [str(i) for i in clust_labels]
cmaps_list   = [color_assign_G214[ct] for ct in clust_labels]

with mpl.rc_context({'figure.figsize': (12, 10)}):
    fig = plot_Multiple(
        labels=clust_labels,
        value_df=adata_st_c2l.obs[clust_col],
        coords=adata_st_c2l.obsm['spatial'],
        cmaps=cmaps_list,
        circle_diameter=2,
        min_color_quantile=0.10,
        max_color_quantile=0.998,
        style='dark_background',
        colorbar_position='right',
        axis_y_flipped=True,
    )
plt.savefig(f'{result_folder}/cell2location_darkbg_{sample}.pdf',
            dpi=300, transparent=False, format='pdf',
            facecolor='black', bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
# ─────────────────────────────────────────────────────────
# 5. 图例（单独，黑底）
# ─────────────────────────────────────────────────────────
n_ct   = len(keep_types_G214)
n_cols = 2
n_rows = (n_ct + n_cols - 1) // n_cols

fig, ax = plt.subplots(figsize=(n_cols * 3, n_rows * 0.35 + 0.3))
fig.patch.set_facecolor('black')
ax.set_facecolor('black')
ax.axis('off')

legend_elements = [
    Line2D([0], [0], marker='o', color='none',
           markerfacecolor=color_assign_G214[ct],
           markeredgecolor='none', markersize=8, label=ct)
    for ct in keep_types_G214
]
legend = ax.legend(
    handles=legend_elements,
    loc='center', frameon=False,
    fontsize=9, ncol=n_cols,
    labelcolor='white',
    handletextpad=0.5, columnspacing=1.0,
)
for text in legend.get_texts():
    text.set_fontfamily('Arial')
plt.tight_layout(pad=0.3)
plt.savefig(f'{result_folder}/legend_{sample}.pdf',
            dpi=300, transparent=False, format='pdf',
            facecolor='black', bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
# ─────────────────────────────────────────────────────────
# 6. 各细胞类型单独热图（每批 8 个）
# ─────────────────────────────────────────────────────────
for batch_i, start in enumerate(range(0, len(keep_types_G214), 8)):
    batch = keep_types_G214[start:start+8]
    fig, axes = plt.subplots(2, 4, figsize=(20, 9))
    axes = axes.flatten()
    for j, ct in enumerate(batch):
        sc.pl.spatial(
            adata_st_c2l,
            img_key=None,
            spot_size=20,
            color=[ct],
            cmap='Reds',
            vmin='p5', vmax='p99',
            ax=axes[j],
            show=False,
            use_raw=False
        )
        axes[j].set_title(ct, fontfamily='Arial', fontsize=10)
    for j in range(len(batch), len(axes)):
        axes[j].set_visible(False)
    plt.suptitle(f'{sample} - Cell type abundance (batch {batch_i+1})',
                 fontsize=12, fontfamily='Arial', y=1.01)
    plt.tight_layout()
    plt.savefig(f'{result_folder}/celltype_heatmap_{sample}_batch{batch_i+1}.pdf',
                dpi=300, transparent=True, format='pdf', bbox_inches='tight')
    plt.show()
    plt.close()

In [ ]:
# ─────────────────────────────────────────────────────────
# 7. UMAP（brain_cluster vs Cell_type_merged）
# ─────────────────────────────────────────────────────────
adata_st.obs['Cell_type_merged'] = adata_st_c2l.obs['Cell_type_merged']

fig, axes = plt.subplots(1, 2, figsize=(10, 6))
sc.pl.umap(adata_st, color='brain_cluster',
           ax=axes[0], show=False, legend_loc='on data',
           legend_fontsize=7, size=20,)
axes[0].set_title(f'{sample} - Brain cluster', fontfamily='Arial', fontsize=11)

sc.pl.umap(adata_st, color='Cell_type_merged',
           palette=palette_G214,
           ax=axes[1], show=False, legend_loc='on data',
           legend_fontsize=7, size=20,)
axes[1].set_title(f'{sample} - Cell type (cell2location)', fontfamily='Arial', fontsize=11)

plt.tight_layout()
plt.savefig(f'{result_folder}/umap_{sample}.pdf',
            dpi=300, transparent=True, format='pdf', bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
# ─────────────────────────────────────────────────────────
# 7. UMAP（brain_cluster vs Cell_type_merged）
# ─────────────────────────────────────────────────────────
adata_st.obs['Cell_type_merged'] = adata_st_c2l.obs['Cell_type_merged']

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
sc.pl.umap(adata_st, color='AS_cluster',
           ax=axes[0], show=False, legend_loc='on data',
           legend_fontsize=10, size=20,)
axes[0].set_title(f'{sample} - Brain cluster', fontfamily='Arial', fontsize=12)

sc.pl.umap(adata_st, color='Cell_type_merged',
           palette=palette_G214,
           ax=axes[1], show=False, legend_loc='on data',
           legend_fontsize=10, size=20,)
axes[1].set_title(f'{sample} - Cell type (cell2location)', fontfamily='Arial', fontsize=12)

plt.tight_layout()
plt.savefig(f'{result_folder}/umap_{sample}.pdf',
            dpi=300, transparent=True, format='pdf', bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
# ─────────────────────────────────────────────────────────
# UMAP：栅格化散点（rasterized=True），其余文字/轴保持矢量
# ─────────────────────────────────────────────────────────
adata_st.obs['Cell_type_merged'] = adata_st_c2l.obs['Cell_type_merged']

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

sc.pl.umap(adata_st, color='AS_cluster',
           ax=axes[0], show=False, legend_loc='on data',
           legend_fontsize=10, size=20)
axes[0].set_title(f'{sample} - Brain cluster', fontfamily='Arial', fontsize=12)

sc.pl.umap(adata_st, color='Cell_type_merged',
           palette=palette_G214,
           ax=axes[1], show=False, legend_loc='on data',
           legend_fontsize=10, size=20)
axes[1].set_title(f'{sample} - Cell type (cell2location)', fontfamily='Arial', fontsize=12)

# 把两个子图的散点集合栅格化
for ax in axes:
    for collection in ax.collections:
        collection.set_rasterized(True)

plt.tight_layout()
plt.savefig(f'{result_folder}/umap_{sample}.pdf',
            dpi=300, transparent=True, format='pdf', bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
sample

In [ ]:
brain_color={
    'Fiber_tracts':      '#1f77b4',
    'FT':      '#1f77b4',
    'Thalamus':      '#ff7f0e',
    'TH':      '#ff7f0e',
    'Cortex_5': '#8c564b',
    'Cortex_3':    '#d62728',
    'Cortex_4':      '#22a884',
    'Cortex_2':    '#2ca02c',
    'Hypothalamus':      '#e377c2',
    'HY':      '#e377c2',
    'Cortex_1':    '#bcbd22',
    'CA1_1':     '#17becf',
    'DG':      '#a6cee3',
    'DG-sg':  '#6baed6', 
    'CA1_2':     '#b2df8a',
    'CA3_1':   '#8fcf73',
    'CA3_2':   '#c9e8b0',
    'Meninges':'#cab2d6',
    'Epithalamus':     '#f3cb29',
    'EPI':     '#f3cb29',
    'RT':      '#a30059',
    'VS':      '#9467bd',
    'HIP':     '#809693',
    'Striatum':     '#9cded6',
    'STR':     '#9cded6',
    'unknown': '#808080'
}

In [ ]:
with mpl.style.context('dark_background'):
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_facecolor('black')
    fig.patch.set_facecolor('black')

    sc.pl.spatial(
        adata_st,
        img_key=None,
        spot_size=26,
        color=['brain_cluster'],
        title=f'{sample}-brain_cluster',
        basis='spatial',
        palette=brain_color,
        ax=ax,
        show=False
    )

    ax.set_title(f'{sample}-brain_cluster', color='white', fontfamily='Arial', fontsize=12)
    legend = ax.get_legend()
    if legend:
        for text in legend.get_texts():
            text.set_color('white')
            text.set_fontfamily('Arial')
        legend.get_frame().set_facecolor('black')
        legend.get_frame().set_edgecolor('none')

    # 散点栅格化
    for collection in ax.collections:
        collection.set_rasterized(True)

    plt.tight_layout()
    plt.savefig(f'{result_folder}/brain_cluster_dark_{sample}.pdf',
                dpi=300, transparent=False, format='pdf',
                facecolor='black', bbox_inches='tight')
    plt.show()
    plt.close()

## 检查

In [ ]:
sp=sc.read("/path/to/project/20260427_AS_analysis/cell2location_result_G21-4/cell2location_map_G21-4/sp.h5ad")
sp

In [ ]:
sp.obs

In [ ]:
# ─────────────────────────────────────────────────────────
# 结果整理 + 丰度统计
# ─────────────────────────────────────────────────────────
sp.obsm['q05_cell_abundance_w_sf']

In [ ]:
print(sp.obsm.keys())

for key in [
    "means_cell_abundance_w_sf",
    "q05_cell_abundance_w_sf",
    "q95_cell_abundance_w_sf",
    "stds_cell_abundance_w_sf"
]:
    print(key, sp.obsm[key].shape)

In [ ]:
abundance = sp.obsm["means_cell_abundance_w_sf"]

print(type(abundance))
print(abundance.shape)

In [ ]:
total_abundance = abundance.sum(axis=0).sort_values(ascending=False)

total_abundance

In [ ]:
plt.figure(figsize=(6, 5))

total_abundance.head(20).sort_values().plot.barh()

plt.xlabel("Total inferred abundance")
plt.ylabel("")
plt.title("Top cell types")

plt.tight_layout()
plt.show()

In [ ]:
sp.obs["total_cell_abundance"] = abundance.sum(axis=1).values

sp.obs["total_cell_abundance"].describe()

In [ ]:
plt.figure(figsize=(5, 4))

plt.hist(
    sp.obs["total_cell_abundance"],
    bins=50
)

plt.xlabel("Total inferred cell abundance per spot")
plt.ylabel("Number of spots")

plt.tight_layout()
plt.show()

In [ ]:
sc.pl.spatial(
    sp,
    color="total_cell_abundance",
    img_key=None,
    spot_size=20,
    cmap="viridis"
)

In [ ]:
for ct in abundance.columns:
    sp.obs[ct] = abundance[ct].values

In [ ]:
sp.obs.columns[-20:]

In [ ]:
sc.pl.spatial(
    sp,
    color=abundance.columns[:6].tolist(),
    img_key=None,
    spot_size=20,
    ncols=3,
    cmap="viridis"
)

In [ ]:
top_ct = total_abundance.head(20).index.tolist()

top_ct

In [ ]:
fig, axes = plt.subplots(
    2, 3,
    figsize=(12, 8)
)

axes = axes.flatten()

xy = sp.obsm["spatial"]

for ax, ct in zip(axes, top_ct):

    values = sp.obs[ct].values

    ax.set_facecolor("black")

    sca = ax.scatter(
        xy[:, 0],
        xy[:, 1],
        c=values,
        s=5,
        cmap="magma",
        edgecolors="none"
    )

    ax.invert_yaxis()
    ax.set_aspect("equal")

    ax.set_title(
        ct,
        fontsize=10
    )

    ax.axis("off")

    plt.colorbar(
        sca,
        ax=ax,
        fraction=0.046,
        pad=0.02
    )

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(
    2, 3,
    figsize=(10, 7)
)

axes = axes.flatten()

xy = sp.obsm["spatial"]

for ax, ct in zip(axes, top_ct):

    values = sp.obs[ct].values

    # 避免极端值影响颜色
    vmax = np.quantile(values, 0.99)

    ax.set_facecolor("black")

    im = ax.scatter(
        xy[:, 0],
        xy[:, 1],
        c=values,
        s=4,
        cmap="magma",
        vmin=0,
        vmax=vmax,
        linewidth=0
    )

    ax.invert_yaxis()
    ax.set_aspect("equal")
    ax.axis("off")

    ax.set_title(
        ct,
        fontsize=10
    )

plt.tight_layout()
plt.show()

# 重新绘图

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import scanpy as sc
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path
from plot_spatial_all import plot_Multiple

# 参数
sp_path = "/path/to/project/20260427_AS_analysis/cell2location_result_G21-4/cell2location_map_G21-4/sp.h5ad"
result_folder = Path("/path/to/project/Fig4/Brain/figures")

min_pct = 0.01
max_color_quantile = 0.994
circle_diameter = 2
figsize = (12, 10)

result_folder.mkdir(parents=True, exist_ok=True)

plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.sans-serif"] = ["Arial"]
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

# 读取解卷积结果
sp = sc.read(sp_path)

factor_names = list(sp.uns["mod"]["factor_names"])
sp.obs[factor_names] = sp.obsm["q05_cell_abundance_w_sf"]

# 统计细胞类型丰度
cell_abundance_sum = sp.obs[factor_names].sum().sort_values(ascending=False)

df_cell_count = pd.DataFrame({"count": cell_abundance_sum})
df_cell_count["percentage"] = (
    df_cell_count["count"] / df_cell_count["count"].sum() * 100
)

# 保留丰度≥1%的细胞类型
keep_types_G214 = df_cell_count.index[
    df_cell_count["percentage"] >= min_pct * 100
].tolist()

print(f"保留 {len(keep_types_G214)} 个细胞类型")
print(df_cell_count.loc[keep_types_G214])

# 原图的颜色
color_assign = {
    "Ext_Thal_1":     "#cac43d",
    "Oligo_1":        "#10976f",
    "Ext_L23":        "#1681ec",
    "Ext_L25":        "#a81e2b",
    "Ext_L56":        "#5a14a5",
    "Ext_Hpc_CA1":    "#f0e442",
    "Ext_Thal_2":     "#b71e75",
    "Ext_Hpc_DG1":    "#009e73",
    "Oligo_2":        "#67c1dd",
    "Ext_L5_1":       "#fd8d3c",
    "Ext_Pir":        "#74c476",
    "Inh_4":          "#ffbe2c",
    "Micro":          "#9e9ac8",
    "Endo":           "#ec7c7f",
    "Ext_Amy_2":      "#8c510a",
    "Inh_1":          "#865123",
    "Nb_2":           "#2ca02c",
    "Ext_Hpc_CA3":    "#56b4e9",
    "Astro_WM":       "#e7969c",
    "Inh_5":          "#7b4173",
    "Inh_Pvalb":      "#ce6dbd",
    "Ext_L6B":        "#3182bd",
    "Inh_Meis2_3":    "#6baed6",
    "Ext_L5_2":       "#e6550d",
    "Ext_L6":         "#31a354",
    "Ext_ClauPyr":    "#756bb1",
    "Inh_6":          "#636363",
    "Astro_HPC":      "#d6616b",
    "Astro_HYPO":     "#8ca252",
    "Astro_THAL_lat": "#bd9e39",
    "Astro_AMY_CTX":  "#ad494a",
    "Ext_Med":        "#a55194",
    "Astro_AMY":      "#6b6ecf",
    "Astro_THAL_hab": "#b5cf6b",
    "Inh_Meis2_4":    "#9c9ede",
    "Inh_Sst":        "#aaaaaa",
}

# 按细胞类型名称分配颜色
color_assign_G214 = {
    ct: color_assign.get(ct, "#aaaaaa")
    for ct in keep_types_G214
}

clust_labels = keep_types_G214
clust_col = [str(ct) for ct in clust_labels]
cmaps_list = [color_assign_G214[ct] for ct in clust_labels]

# 保存图中实际使用的丰度和颜色
df_plot = df_cell_count.loc[keep_types_G214].copy()
df_plot["color"] = [color_assign_G214[ct] for ct in df_plot.index]
df_plot.to_csv(result_folder / "G21-4_celltype_abundance_colors.csv")

# 黑底混色图
with mpl.rc_context({"figure.figsize": figsize}):
    plot_Multiple(
        labels=clust_labels,
        value_df=sp.obs[clust_col],
        coords=sp.obsm["spatial"],
        cmaps=cmaps_list,
        circle_diameter=circle_diameter,
        min_color_quantile=0.10,
        max_color_quantile=max_color_quantile,
        style="dark_background",
        colorbar_position="right",
        axis_y_flipped=True,
    )

output = result_folder / "cell2location_darkbg_G21-4.pdf"

plt.savefig(
    output,
    dpi=300,
    transparent=False,
    format="pdf",
    facecolor="black",
    bbox_inches="tight",
)

plt.show()
plt.close()

print("保存至:", output)

In [ ]:
# ─────────────────────────────────────────────────────────
# 1. 黑底矢量图
# ─────────────────────────────────────────────────────────
output_black = result_folder / "cell2location_darkbg_G21-4_vector.pdf"

with mpl.rc_context({"figure.figsize": figsize}):
    fig, _, _ = plot_Multiple(
        labels=clust_labels,
        value_df=sp.obs[clust_col].copy(),
        coords=sp.obsm["spatial"],
        cmaps=cmaps_list,
        circle_diameter=circle_diameter,
        min_color_quantile=0.10,
        max_color_quantile=max_color_quantile,
        style="dark_background",
        colorbar_position="right",
        axis_y_flipped=True,
    )

# 取消plot_Multiple内部设置的栅格化
for ax in fig.axes:
    ax.set_rasterized(False)
    for collection in ax.collections:
        collection.set_rasterized(False)

fig.savefig(
    output_black,
    format="pdf",
    transparent=False,
    facecolor="black",
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

print("黑底矢量图保存至:", output_black)


# ─────────────────────────────────────────────────────────
# 2. 白底矢量图
# ─────────────────────────────────────────────────────────
output_white = result_folder / "cell2location_whitebg_G21-4_vector.pdf"

with mpl.rc_context({"figure.figsize": figsize}):
    fig, _, _ = plot_Multiple(
        labels=clust_labels,
        value_df=sp.obs[clust_col].copy(),
        coords=sp.obsm["spatial"],
        cmaps=cmaps_list,
        circle_diameter=circle_diameter,
        min_color_quantile=0.10,
        max_color_quantile=max_color_quantile,
        style="fast",               # 白底、黑色文字
        colorbar_position="right",
        axis_y_flipped=True,
    )

# 取消栅格化
for ax in fig.axes:
    ax.set_rasterized(False)
    for collection in ax.collections:
        collection.set_rasterized(False)

fig.savefig(
    output_white,
    format="pdf",
    transparent=False,
    facecolor="white",
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

print("白底矢量图保存至:", output_white)

# th

In [ ]:
sp_th = sc.read(
    "/path/to/project/ref_datas/"
    "2023-Allen_WMB/TH/results/G21-4_TH/sp_cell2location.h5ad"
)
sp_th

In [ ]:
all_cell_types = list(sp_th.uns["mod"]["factor_names"])

clust_labels = [
    "151 TH Prkcd Grin2c Glut",
    "152 RE-Xi Nox4 Glut",
    "153 MG-POL-SGN Nts Glut",
    "154 PF Fzd5 Glut",
]

abundance_df = sp_th.obsm["q05_cell_abundance_w_sf"]

if not isinstance(abundance_df, pd.DataFrame):
    abundance_df = pd.DataFrame(
        abundance_df,
        index=sp_th.obs_names,
        columns=all_cell_types,
    )
else:
    abundance_df = abundance_df.copy()
    abundance_df.index = sp_th.obs_names

value_df = abundance_df[clust_labels].astype(float)

print(value_df.shape)
display(value_df.describe().T)

In [ ]:
fig, _, _ = plot_Multiple(
    labels=clust_labels,
    value_df=value_df,
    coords=sp_th.obsm["spatial"],
    cmaps=cmaps_list,
    circle_diameter=circle_diameter,
    min_color_quantile=0.10,
    max_color_quantile=max_color_quantile,
    style="dark_background",
    colorbar_position="right",
    axis_y_flipped=True,
)

In [ ]:
# ─────────────────────────────────────────────────────────
# 参数
# ─────────────────────────────────────────────────────────
output_black = result_folder / "cell2location_TH_darkbg_G21-4_vector.pdf"
output_white = result_folder / "cell2location_TH_whitebg_G21-4_vector.pdf"


# ─────────────────────────────────────────────────────────
# 1. 黑底矢量图
# ─────────────────────────────────────────────────────────
with mpl.rc_context({"figure.figsize": figsize}):
    fig, _, _ = plot_Multiple(
        labels=clust_labels,
        value_df=value_df,
        coords=sp_th.obsm["spatial"],
        cmaps=cmaps_list,
        circle_diameter=circle_diameter,
        min_color_quantile=0.10,
        max_color_quantile=max_color_quantile,
        style="dark_background",
        colorbar_position="right",
        axis_y_flipped=True,
    )

# 取消栅格化
for ax in fig.axes:
    ax.set_rasterized(False)
    for collection in ax.collections:
        collection.set_rasterized(False)

fig.savefig(
    output_black,
    format="pdf",
    transparent=False,
    facecolor="black",
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

print("黑底矢量图保存至:", output_black)


# ─────────────────────────────────────────────────────────
# 2. 白底矢量图
# ─────────────────────────────────────────────────────────
with mpl.rc_context({"figure.figsize": figsize}):
    fig, _, _ = plot_Multiple(
        labels=clust_labels,
        value_df=value_df,
        coords=sp_th.obsm["spatial"],
        cmaps=cmaps_list,
        circle_diameter=circle_diameter,
        min_color_quantile=0.10,
        max_color_quantile=max_color_quantile,
        style="fast",
        colorbar_position="right",
        axis_y_flipped=True,
    )

# 取消栅格化
for ax in fig.axes:
    ax.set_rasterized(False)
    for collection in ax.collections:
        collection.set_rasterized(False)

fig.savefig(
    output_white,
    format="pdf",
    transparent=False,
    facecolor="white",
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

print("白底矢量图保存至:", output_white)

# 000